<div align="center">

# YAPAY ZEKÂ DESTEKLİ İLAÇ TASARIMI

**Erzincan Binali Yıldırım Üniversitesi — Sürekli Eğitim Uygulama ve Araştırma Merkezi**

28–29 Kasım 2026 · ML tabanlı QSAR uygulamalı eğitimi

</div>

---

### Eğitim sorumlusu

**Dr. Öğr. Üyesi Ekrem Yaşar**
Erzincan Binali Yıldırım Üniversitesi, Tıp Fakültesi, Biyofizik Anabilim Dalı

📧 `ey.biophysics@gmail.com`

---

# Oturum 6 — Yeni Molekülleri Taramak ve SDF Üretmek

**Girdi:** Oturum 3, 4 ve 5'in modelleri ve AD parametreleri
**Çıktı:** sıralanmış aday listesi + tek birleşik SDF + molekül başına ayrı SDF

Son oturum. Buraya kadar bir model kurduk, ayarladık, doğruladık ve açıkladık.
Şimdi onu asıl işi için kullanacağız: **daha önce test edilmemiş moleküller
hakkında tahmin üretmek.**

## Tam iş akışı

```
Tarama kütüphanesi (binlerce molekül)
          │
          ▼
  Özellik üretimi  (Oturum 2 ile AYNI yöntem)
          │
          ├──► SINIFLANDIRICI  ──► aktif mi? + olasılık
          ├──► REGRESYON       ──► tahmin pIC50
          └──► UYGULANABİLİRLİK ALANI ──► bu tahmine güvenilir mi?
                      │
                      ▼
              Gruplama ve sıralama
                      │
        ┌─────────────┴─────────────┐
        ▼                           ▼
  tek birleşik SDF          molekül başına ayrı SDF
```

**Niçin iki model birlikte?** Sanal taramada tipik akış ikisini birleştirir:
**sınıflandırıcı** hızlı bir elek görevi görür, **regresyon** modeli kalanları
potense göre sıralar. Üçüncü bileşen **AD kontrolü**, hangi tahminlere
güvenebileceğimizi işaretler.

## Neden SDF?

**SDF**, kimyasal yapıyı **ve** ona bağlı özellikleri birlikte taşıyan standart
dosya biçimidir. CSV'den farkı: molekülün **3B koordinatlarını** ve istediğiniz
kadar etiketli alanı içerebilir.

Bu, iş akışının bir sonraki adımına köprüdür: ürettiğimiz SDF dosyaları
doğrudan **moleküler doklama** (AutoDock Vina, Glide) ya da görselleştirme
(PyMOL, ChimeraX) programlarına verilebilir.

| Biçim | Ne için? |
|---|---|
| **Tek birleşik SDF** | Bütün adayları bir arada doklamak |
| **Molekül başına ayrı SDF** | Tek tek inceleme, paralel doklama işleri |

⏱️ Yaklaşık süre: 8–15 dakika

---
## 1. Yapılandırma

**`KUTUPHANE_KAYNAGI`** üç seçenek sunar:

- `"depo"` — kurs deposundaki klinik ilaç kütüphanesi, faz 1–4 (varsayılan).
  Bu bir **ilaç yeniden konumlandırma** (drug repurposing) alıştırmasıdır:
  zaten onaylanmış, güvenlik profili bilinen moleküller arasında bu hedefe
  etki edebilecek olanları aramak.
- `"chembl"` — ChEMBL'den canlı indirme.
- `"kendi"` — kendi SMILES listenizi `KENDI_SMILES` içine yazın.

**`UC_BOYUT`** açıkken SDF dosyaları 3B koordinat içerir (doklama için
gereklidir) ama üretimi yavaştır; bu yüzden yalnızca ilk `TOP_N` molekül için
yapılır.

In [ ]:
# ── Hedef ─────────────────────────────────────────────────────────────────────
TARGET_ADI  = "NTRK1"
HEDEF_KOLON = "pIC50"

# Aktiflik esigi Oturum 1'de secilmistir; asagida dosyadan okunuyor.

# ── Google Drive ANA klasoru (Oturum 1-5 ile AYNI) ───────────────────────────
DRIVE_PROJE_KLASORU = "/content/drive/MyDrive/EBYUSEM2026_MLQSAR"
YEREL_KLASOR        = "EBYUSEM2026_MLQSAR"
DEPO_RAW            = "https://raw.githubusercontent.com/eygpcr/ebyusem2026-mlqsar/main"

# ── Onceki oturumlarin ciktilari ─────────────────────────────────────────────
AD_REG_MODEL = "ntrk1_en_iyi_regresyon_modeli.joblib"      # Oturum 3
AD_SNF_MODEL = "ntrk1_en_iyi_siniflandirma_modeli.joblib"  # Oturum 4
AD_DESC      = "ntrk1_descriptors.csv"                     # Oturum 2
AD_AD_PARAM  = "ntrk1_ad_parametreleri.json"               # Oturum 5

# ── Tarama kutuphanesi: "depo" | "chembl" | "kendi" ──────────────────────────
KUTUPHANE_KAYNAGI = "depo"
AD_KUTUPHANE      = "tarama_kutuphanesi.csv"
CHEMBL_FAZLAR     = (4, 3, 2, 1)   # "chembl" secilirse indirilecek fazlar

# Kendi listenizi kullanmak icin KUTUPHANE_KAYNAGI = "kendi" yapin:
KENDI_SMILES = [
    "CC(=O)Oc1ccccc1C(=O)O",                      # aspirin
    "CN1C=NC2=C1C(=O)N(C)C(=O)N2C",               # kafein
    "CC(C)Cc1ccc(cc1)C(C)C(=O)O",                 # ibuprofen
]

# ── Aday secimi ──────────────────────────────────────────────────────────────
TOP_N          = 25      # kac aday secilecek
OLASILIK_ESIGI = None    # None -> Oturum 4'te secilen karar esigi kullanilir
B_ILE_TAMAMLA  = True    # A grubu TOP_N'i doldurmazsa B grubundan tamamla

# ── SDF ayarlari ─────────────────────────────────────────────────────────────
UC_BOYUT       = True        # 3B koordinat uret (doklama icin)
RASTGELE_TOHUM = 42

print(f"Hedef              : {TARGET_ADI}")
print(f"Kutuphane kaynagi  : {KUTUPHANE_KAYNAGI}")
print(f"Secilecek aday     : {TOP_N}")
print(f"3B koordinat       : {UC_BOYUT}")

In [ ]:
import json
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJE, ORTAM, COLAB = Path(DRIVE_PROJE_KLASORU), "Colab + Drive", True
except (ImportError, ModuleNotFoundError):
    PROJE, ORTAM, COLAB = Path(YEREL_KLASOR), "Yerel", False

KLASOR = {
    1: PROJE / "01_veri",
    2: PROJE / "02_ozellikler",
    3: PROJE / "03_regresyon",
    4: PROJE / "04_siniflandirma",
    5: PROJE / "05_analiz",
    6: PROJE / "06_adaylar",
}
for k in KLASOR.values():
    (k / "gorseller").mkdir(parents=True, exist_ok=True)

BU     = KLASOR[6]
GORSEL = BU / "gorseller"

# SDF ciktilari bu oturumun klasorunde
SDF_KLASOR  = BU / "sdf"
SDF_TEK_TEK = SDF_KLASOR / "molekul_basina"
SDF_TEK_TEK.mkdir(parents=True, exist_ok=True)

# Oturum 1'de secilen esik
yol_esik = KLASOR[1] / "esik_secimi.json"
ESIK_SECIMI = (json.loads(yol_esik.read_text(encoding="utf-8"))
               if yol_esik.exists() else None)
ESIK = ESIK_SECIMI["esik"] if ESIK_SECIMI else None

print(f"Ortam       : {ORTAM}")
print(f"Ana klasor  : {PROJE}")
print(f"Bu oturum   : {BU}")
print(f"SDF klasoru : {SDF_KLASOR}")
if ESIK_SECIMI:
    print(f"Oturum 1 esigi: mod '{ESIK_SECIMI['mod']}', esik {ESIK}")

In [ ]:
!pip install rdkit -q
print("Kurulum tamam.")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import shutil
import time
import zipfile

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem, Descriptors, Draw, rdFingerprintGenerator
from rdkit.ML.Descriptors.MoleculeDescriptors import MolecularDescriptorCalculator
from sklearn.preprocessing import StandardScaler

RDLogger.DisableLog("rdApp.*")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 70)

print("Kutuphaneler yuklendi.")

---
## 2. Eğitilmiş modelleri ve AD parametrelerini yükleme

Oturum 3, 4 ve 5'in çıktılarını geri yüklüyoruz. Modeller `Pipeline` olarak
kaydedildiği için ölçekleyici de içlerinde gelir — ayrıca ölçekleme yapmamız
gerekmez.

In [ ]:
def gerekli_dosya(ad, oturum):
    """Onceki bir oturumun ciktisini bulur; yoksa anlasilir bir hata verir."""
    yol = KLASOR[oturum] / ad
    if not yol.exists():
        raise FileNotFoundError(
            f"'{ad}' bulunamadi.\n"
            f"Bu dosya Oturum {oturum}'in ciktisidir. Butun oturumlari (01-05) "
            f"ayni Drive klasorunu kullanarak calistirdiginizdan emin olun.\n"
            f"Beklenen yol: {yol}"
        )
    return yol


paket_reg = joblib.load(gerekli_dosya(AD_REG_MODEL, 3))
print(f"Regresyon modeli     : {paket_reg['model_adi']} "
      f"({paket_reg['ozellik_seti']})")
print(f"  Test R2={paket_reg['test_r2']:.4f}  "
      f"Test RMSE={paket_reg['test_rmse']:.4f}")

paket_snf = None
if (KLASOR[4] / AD_SNF_MODEL).exists():
    paket_snf = joblib.load(KLASOR[4] / AD_SNF_MODEL)
    print(f"\nSiniflandirma modeli : {paket_snf['model_adi']} "
          f"({paket_snf['ozellik_seti']})")
    print(f"  Test ROC-AUC={paket_snf['test_roc_auc']:.4f}  "
          f"Test MCC={paket_snf['test_mcc']:.4f}")
    print(f"  Karar esigi={paket_snf.get('karar_esigi', 0.5):.3f}")
else:
    print("\n!! Siniflandirma modeli bulunamadi; yalnizca regresyon "
          "kullanilacak.")

ad_param = None
if (KLASOR[5] / AD_AD_PARAM).exists():
    ad_param = json.loads((KLASOR[5] / AD_AD_PARAM).read_text(encoding="utf-8"))
    print(f"\nAD parametreleri     : h* = {ad_param['kaldirac']['h_kritik']:.4f}, "
          f"Tanimoto esigi = {ad_param['tanimoto']['esik']:.3f}")
else:
    print("\n!! AD parametreleri bulunamadi (Oturum 5); AD kontrolu "
          "bu oturumda yeniden kurulacak.")

### Eğitim verisini geri yükleme

Uygulanabilirlik alanı hesabı için modelin **eğitildiği** veriye ihtiyacımız
var: kaldıraç matrisi ve Tanimoto karşılaştırması için.

In [ ]:
tablo_egitim = pd.read_csv(gerekli_dosya(AD_DESC, 2))
ozellik_adlari = paket_reg["ozellik_adlari"]

egitim_idx = np.array(paket_reg["egitim_indeksleri"])
X_egitim_tam = tablo_egitim[ozellik_adlari].iloc[egitim_idx]
y_egitim     = tablo_egitim[HEDEF_KOLON].iloc[egitim_idx]
smiles_egitim = tablo_egitim["Smiles"].iloc[egitim_idx]

# Kaldirac altyapisi (olcekleyici YALNIZCA egitim verisine uydurulur)
olcekleyici_ad = StandardScaler().fit(X_egitim_tam.values)
Xe_olcekli = olcekleyici_ad.transform(X_egitim_tam.values)
XtX_ters = np.linalg.pinv(Xe_olcekli.T @ Xe_olcekli)

n_egitim, p_ozellik = Xe_olcekli.shape
h_kritik = (ad_param["kaldirac"]["h_kritik"] if ad_param
            else 3.0 * (p_ozellik + 1) / n_egitim)
tanimoto_esigi = ad_param["tanimoto"]["esik"] if ad_param else 0.25

print(f"Egitim molekulu : {n_egitim}")
print(f"Ozellik sayisi  : {p_ozellik}")
print(f"h* (kritik)     : {h_kritik:.4f}")
print(f"Tanimoto esigi  : {tanimoto_esigi:.4f}")

---
## 3. Tarama kütüphanesini yüklemek

Kütüphanemiz ChEMBL'den alınmış **klinik aşamaya gelmiş moleküllerdir**
(faz 1, 2, 3 ve onaylı faz 4). Bu seçim gerçek bir araştırma yaklaşımına
karşılık gelir: **ilaç yeniden konumlandırma** (drug repurposing).

Mantığı şudur: bu moleküller insanlarda zaten denenmiştir, güvenlik profilleri
kısmen ya da tamamen bilinir. Biri bizim hedefimize beklenmedik biçimde etki
ediyorsa, geliştirme süreci sıfırdan başlayan bir molekülden **çok daha hızlı**
ilerleyebilir.

**Neden yalnızca onaylı ilaçlar değil?** Faz 4 kütüphanesi küçüktür ve kinaz
inhibitörü bakımından fakirdir. Faz 1–3'te ise çok sayıda kinaz inhibitörü
bulunur — tam da modelimizin tanıdığı kimyasal uzay. Yeniden konumlandırma
literatürü de klinik aşamadaki bileşenleri kapsar.

> **`Egitimde_var` kolonu.** Bazı moleküller eğitim setimizde de bulunuyor
> (çünkü ChEMBL'de bu hedefe karşı ölçülmüşler). Bunları **silmiyoruz,
> işaretliyoruz** — bilinen aktiflerin yüksek tahmin alması modelin iyi bir
> sağlık kontrolüdür (Bölüm 7). Aday listesinden ise çıkarıyoruz, çünkü onlar
> yeni bir keşif olmaz.

In [ ]:
if KUTUPHANE_KAYNAGI == "depo":
    yol_yerel = BU / AD_KUTUPHANE
    if yol_yerel.exists():
        kutuphane = pd.read_csv(yol_yerel)
        print(f"Drive'dan okundu: {yol_yerel.name}")
    else:
        url = f"{DEPO_RAW}/03_veri/{AD_KUTUPHANE}"
        kutuphane = pd.read_csv(url)
        kutuphane.to_csv(yol_yerel, index=False)   # bir daha inmesin
        print(f"Depodan okundu ve Drive'a kaydedildi: {yol_yerel.name}")

elif KUTUPHANE_KAYNAGI == "chembl":
    print(f"ChEMBL'den klinik asama molekulleri indiriliyor (faz "
          f"{', '.join(str(f) for f in CHEMBL_FAZLAR)})...")
    kayitlar = []
    for faz in CHEMBL_FAZLAR:
        print(f"  faz {faz}:")
        offset = 0
        while offset < 8000:
            try:
                yanit = requests.get(
                    "https://www.ebi.ac.uk/chembl/api/data/molecule.json",
                    params={"max_phase": faz, "limit": 1000, "offset": offset,
                            "only": ("molecule_chembl_id,pref_name,"
                                     "molecule_structures,molecule_type,"
                                     "max_phase")},
                    timeout=120)
                yanit.raise_for_status()
                sayfa = yanit.json().get("molecules", [])
            except Exception as hata:
                print(f"    offset {offset}: alinamadi "
                      f"({type(hata).__name__}), atlandi")
                offset += 1000
                continue
            for m in sayfa:
                yapi = m.get("molecule_structures") or {}
                if yapi.get("canonical_smiles"):
                    kayitlar.append({
                        "molecule_chembl_id": m.get("molecule_chembl_id"),
                        "Isim": (m.get("pref_name") or "").title() or None,
                        "Molekul_tipi": m.get("molecule_type"),
                        "max_phase": faz,
                        "Smiles": yapi["canonical_smiles"],
                    })
            print(f"    offset {offset:>5}: +{len(sayfa)}")
            if len(sayfa) < 1000:
                break
            offset += 1000
    kutuphane = pd.DataFrame(kayitlar)
    kutuphane = kutuphane[kutuphane["Molekul_tipi"] == "Small molecule"]

elif KUTUPHANE_KAYNAGI == "kendi":
    kutuphane = pd.DataFrame({
        "molecule_chembl_id": [f"KENDI_{i + 1}" for i in range(len(KENDI_SMILES))],
        "Isim": [None] * len(KENDI_SMILES),
        "Smiles": KENDI_SMILES,
    })
    print(f"Kendi listeniz kullaniliyor: {len(kutuphane)} molekul")

else:
    raise ValueError(f"Bilinmeyen KUTUPHANE_KAYNAGI: {KUTUPHANE_KAYNAGI}")

print(f"\nHam kutuphane boyutu: {len(kutuphane)}")
kutuphane.head()

In [ ]:
# Kanoniklestirme ve gecerlilik kontrolu (Oturum 1'deki hattin aynisi)
def kanonik(smi):
    if not isinstance(smi, str) or not smi:
        return None
    mol = Chem.MolFromSmiles(smi)
    return Chem.MolToSmiles(mol, canonical=True) if mol else None


once = len(kutuphane)
kutuphane["Smiles"] = kutuphane["Smiles"].apply(kanonik)
kutuphane = kutuphane.dropna(subset=["Smiles"]).drop_duplicates(subset="Smiles")
kutuphane = kutuphane.reset_index(drop=True)

# Egitim setiyle ortusme
egitim_kumesi = set(smiles_egitim)
if "Egitimde_var" not in kutuphane.columns:
    kutuphane["Egitimde_var"] = kutuphane["Smiles"].isin(egitim_kumesi).astype(int)
else:
    kutuphane["Egitimde_var"] = kutuphane["Smiles"].isin(egitim_kumesi).astype(int)

print(f"Gecerli ve benzersiz molekul : {len(kutuphane)}  (elenen: {once - len(kutuphane)})")
print(f"Egitim setinde de bulunan    : {int(kutuphane['Egitimde_var'].sum())}")
print(f"Yeni (hic gorulmemis)        : {int((1 - kutuphane['Egitimde_var']).sum())}")

### Kütüphanede hangi fazdan kaç ilaç var?

Yeniden konumlandırma için elimizdeki havuzun **klinik olgunluğunu** bilmek
gerekir. Faz numarası yükseldikçe molekül hakkında bilinenler artar:

| Faz | Anlamı | Yeniden konumlandırma açısından |
|---|---|---|
| **1** | İnsanda ilk deneme — güvenlik ve doz | En çok molekül, en az bilgi |
| **2** | Küçük hasta grubunda etkinlik | Güvenlik verisi var |
| **3** | Geniş ölçekli doğrulama | Kapsamlı klinik veri |
| **4** | **Onaylanmış ilaç** | Ruhsatlı; en hızlı yeniden konumlandırılabilir |

Faz 4'teki bir molekül bulursanız yol en kısadır: ilaç zaten piyasadadır.
Faz 1–3'teki bir molekül de değerlidir — geliştirme süreci başlamıştır ve
insan verisi mevcuttur.

In [ ]:
FAZ_ACIKLAMA = {
    1: "Faz 1 — insanda ilk deneme (guvenlik, doz)",
    2: "Faz 2 — kucuk hasta grubunda etkinlik",
    3: "Faz 3 — genis olcekli dogrulama",
    4: "Faz 4 — ONAYLANMIS ilac",
}

if "max_phase" in kutuphane.columns:
    faz_sayim = (kutuphane["max_phase"].astype(int).value_counts()
                 .sort_index().rename_axis("Faz").rename("Molekul").to_frame())
    faz_sayim["Yuzde"] = (100 * faz_sayim["Molekul"] / len(kutuphane)).round(1)
    faz_sayim["Aciklama"] = [FAZ_ACIKLAMA.get(int(f), "") for f in faz_sayim.index]

    print(f"TARAMA KUTUPHANESI — {len(kutuphane)} molekul")
    print("=" * 76)
    print(faz_sayim.to_string())

    print()
    print("Kumulatif (faz X ve ustu)")
    print("-" * 40)
    for faz in sorted(faz_sayim.index):
        n = int(faz_sayim.loc[faz_sayim.index >= faz, "Molekul"].sum())
        print(f"  Faz {int(faz)} ve ustu : {n:>6} molekul")

    # --- Grafik ---
    fig, eksen = plt.subplots(figsize=(9, 4.2))
    renkler = ["#aab7c4", "#7f9bb5", "#4a7fa8", "#27ae60"]
    cubuk = eksen.bar([f"Faz {int(f)}" for f in faz_sayim.index],
                      faz_sayim["Molekul"],
                      color=[renkler[min(int(f) - 1, 3)] for f in faz_sayim.index],
                      edgecolor="white")
    for c, (n, y) in zip(cubuk, zip(faz_sayim["Molekul"], faz_sayim["Yuzde"])):
        eksen.text(c.get_x() + c.get_width() / 2, n, f"{n:,}\n(%{y})",
                   ha="center", va="bottom", fontsize=9.5)
    eksen.set_ylabel("Molekul sayisi")
    eksen.set_title("Tarama kutuphanesinin klinik faz dagilimi")
    eksen.set_ylim(0, faz_sayim["Molekul"].max() * 1.22)
    eksen.grid(alpha=0.3, axis="y", linestyle="--")
    plt.tight_layout()
    plt.savefig(GORSEL / "01_klinik_faz_dagilimi.png", dpi=150,
                bbox_inches="tight")
    plt.show()

    onayli = int(faz_sayim.loc[faz_sayim.index == 4, "Molekul"].sum())
    print(f"Onaylanmis (faz 4) ilac sayisi : {onayli}")
    print(f"Klinik asamadaki diger molekul : {len(kutuphane) - onayli}")
else:
    print("Kutuphanede 'max_phase' kolonu yok; faz dagilimi gosterilemiyor.")
    print(f"Toplam molekul: {len(kutuphane)}")

---
## 4. Kütüphane için özellik üretimi

**Kritik kural:** Tahmin yapacağımız moleküllerin özellikleri, modelin
eğitildiği özelliklerle **tam olarak aynı yöntemle** üretilmelidir. Aynı
betimleyiciler, aynı sırada, aynı isimlerle.

Bu yüzden Oturum 2'deki hesaplamayı birebir tekrarlıyor, ardından modelin
beklediği kolonları (`ozellik_adlari`) seçiyoruz.

Bazı moleküller için bir betimleyici hesaplanamayabilir (`NaN`/`inf`). Bu
moleküller için güvenilir tahmin üretilemez, bu yüzden onları raporlayarak
çıkarıyoruz.

In [ ]:
butun_betimleyiciler = [ad for ad, _ in Descriptors._descList]
hesaplayici = MolecularDescriptorCalculator(butun_betimleyiciler)

moller = [Chem.MolFromSmiles(s) for s in kutuphane["Smiles"]]
gecerli = [m is not None for m in moller]
kutuphane = kutuphane[gecerli].reset_index(drop=True)
moller = [m for m in moller if m is not None]

print(f"Betimleyiciler hesaplaniyor ({len(moller)} molekul)...")
baslangic = time.time()


def hesapla(mol):
    try:
        return hesaplayici.CalcDescriptors(mol)
    except Exception:
        return (np.nan,) * len(butun_betimleyiciler)


X_ham = pd.DataFrame([hesapla(m) for m in moller],
                     columns=butun_betimleyiciler)
print(f"  {time.time() - baslangic:.1f} s")

# Modelin bekledigi kolonlari sec
eksik = [k for k in ozellik_adlari if k not in X_ham.columns]
if eksik:
    raise KeyError(f"Bu betimleyiciler hesaplanamadi: {eksik}")

X_kutuphane = X_ham[ozellik_adlari].apply(pd.to_numeric, errors="coerce")
X_kutuphane = X_kutuphane.replace([np.inf, -np.inf], np.nan)

# Oturum 2'deki ayni guvence: float32'ye sigmayacak kadar buyuk degerler de
# gecersiz sayilir (bkz. Oturum 2, Bolum 9 - 'Ipc' ornegi).
X_kutuphane = X_kutuphane.mask(X_kutuphane.abs() > 1e30)

sorunlu = X_kutuphane.isna().any(axis=1)
if sorunlu.any():
    print(f"\n{int(sorunlu.sum())} molekulde gecersiz betimleyici degeri var; "
          f"bunlar cikariliyor.")
    kutuphane = kutuphane[~sorunlu.values].reset_index(drop=True)
    moller = [m for m, s in zip(moller, sorunlu.values) if not s]
    X_kutuphane = X_kutuphane[~sorunlu.values].reset_index(drop=True)

print(f"\nTahmine hazir molekul : {len(X_kutuphane)}")
print(f"Ozellik sayisi        : {X_kutuphane.shape[1]}  (model ile ayni)")

---
## 5. Tahmin: iki model birlikte

Şimdi her iki modeli de çalıştırıyoruz:

- **Regresyon** → tahmin edilen `pIC50` (ve nM karşılığı)
- **Sınıflandırma** → aktif olma olasılığı ve sınıf kararı

İkisi her zaman birbirini doğrulamaz; uyuşmazlık olduğunda bu bilgidir.
Örneğin regresyon yüksek potens tahmin ederken sınıflandırıcı düşük olasılık
veriyorsa, molekül karar sınırının yakınındadır ve tahmine daha az
güvenilmelidir.

In [ ]:
pipeline_reg = paket_reg["pipeline"]
tahmin_pic50 = pipeline_reg.predict(X_kutuphane)

sonuc = kutuphane.copy()
sonuc[f"tahmin_{HEDEF_KOLON}"] = tahmin_pic50
sonuc["tahmin_nM"] = 10 ** (9 - tahmin_pic50)

if paket_snf is not None:
    pipeline_snf = paket_snf["pipeline"]
    ozellik_snf = paket_snf["ozellik_adlari"]
    X_snf = X_ham[ozellik_snf].apply(pd.to_numeric, errors="coerce")
    X_snf = X_snf.replace([np.inf, -np.inf], np.nan)
    X_snf = X_snf.loc[X_kutuphane.index]

    olasilik = pipeline_snf.predict_proba(X_snf)[:, 1]
    karar_esigi = (OLASILIK_ESIGI if OLASILIK_ESIGI is not None
                   else paket_snf.get("karar_esigi", 0.5))
    sonuc["aktif_olasiligi"] = olasilik
    sonuc["siniflandirici_aktif"] = (olasilik >= karar_esigi).astype(int)
    print(f"Siniflandirici karar esigi: {karar_esigi:.3f}")
    print(f"  'Aktif' denen molekul   : {int(sonuc['siniflandirici_aktif'].sum())}"
          f" / {len(sonuc)}")
else:
    karar_esigi = None
    sonuc["aktif_olasiligi"] = np.nan
    sonuc["siniflandirici_aktif"] = np.nan

print(f"\nTahmin edilen {HEDEF_KOLON} dagilimi")
print("-" * 50)
print(sonuc[f"tahmin_{HEDEF_KOLON}"].describe().round(3).to_string())

---
## 6. Uygulanabilirlik alanı kontrolü

Her tahminin yanına bir **güven etiketi** koyuyoruz. Oturum 5'te kurduğumuz iki
ölçütü birlikte kullanıyoruz:

1. **Kaldıraç** `h < h*` → özellik uzayında eğitim verisinin içinde
2. **Tanimoto** benzerliği ≥ eşik → yapısal olarak eğitim verisine benziyor

Bir molekülü **AD içinde** saymak için ikisini de geçmesi gerekir. Muhafazakâr
bir tercih: "emin değilsem güvenmem".

> **Niçin bu kadar titizleniyoruz?** Klinik ilaç kütüphanesi, eğitim setimizden
> (kinaz inhibitörleri) çok farklı sınıflar içerir: antibiyotikler, steroidler,
> peptitler... Model bunların hiçbirini görmedi. AD kontrolü olmadan, bir
> antibiyotik için üretilen `pIC50 = 7.5` sayısına inanma riskimiz olurdu.

In [ ]:
# --- Kaldirac ---
X_kutuphane_olcekli = olcekleyici_ad.transform(X_kutuphane.values)
h_kutuphane = np.einsum("ij,jk,ik->i", X_kutuphane_olcekli, XtX_ters,
                        X_kutuphane_olcekli)
sonuc["kaldirac"] = h_kutuphane
sonuc["kaldirac_ok"] = (h_kutuphane <= h_kritik)

# --- Tanimoto benzerligi ---
morgan = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

print("Egitim seti parmak izleri...")
iz_egitim = []
for s in smiles_egitim:
    m = Chem.MolFromSmiles(s)
    if m is not None:
        iz_egitim.append(morgan.GetFingerprint(m))

print("Kutuphane benzerlikleri hesaplaniyor...")
en_benzer, en_benzer_smiles = [], []
for mol in moller:
    iz = morgan.GetFingerprint(mol)
    benzerlikler = DataStructs.BulkTanimotoSimilarity(iz, iz_egitim)
    konum = int(np.argmax(benzerlikler))
    en_benzer.append(benzerlikler[konum])
    en_benzer_smiles.append(smiles_egitim.iloc[konum])

sonuc["en_yuksek_tanimoto"] = en_benzer
sonuc["en_benzer_egitim_smiles"] = en_benzer_smiles
sonuc["tanimoto_ok"] = sonuc["en_yuksek_tanimoto"] >= tanimoto_esigi

# --- Birlesik AD karari ---
sonuc["AD_icinde"] = sonuc["kaldirac_ok"] & sonuc["tanimoto_ok"]

print("\nUygulanabilirlik alani ozeti")
print("=" * 62)
print(f"  Kaldirac testini gecen (h <= {h_kritik:.3f})     : "
      f"{int(sonuc['kaldirac_ok'].sum()):>5} / {len(sonuc)}")
print(f"  Tanimoto testini gecen (>= {tanimoto_esigi:.3f})    : "
      f"{int(sonuc['tanimoto_ok'].sum()):>5} / {len(sonuc)}")
print(f"  IKISINI de gecen (AD icinde)             : "
      f"{int(sonuc['AD_icinde'].sum()):>5} / {len(sonuc)}"
      f"  (%{100 * sonuc['AD_icinde'].mean():.1f})")
print(f"\nBenzerlik dagilimi (en yuksek Tanimoto):")
print(sonuc["en_yuksek_tanimoto"].describe().round(3).to_string())

### Sağlık kontrolü: bilinen NTRK inhibitörleri yüksek tahmin alıyor mu?

Bu, modelimizin en ikna edici sınavıdır.

Kütüphanemizde **gerçek NTRK inhibitörleri** bulunuyor: **larotrektinib** ve
**entrektinib** (FDA onaylı), **selitrektinib** ve **repotrektinib** (ikinci
nesil), ayrıca **krizotinib** gibi NTRK'ye de etki eden çoklu-kinaz
inhibitörleri.

Model bunları kütüphanenin en üstünde sıralıyorsa, öğrendiği şeyin gerçek
olduğuna dair güçlü bir kanıttır. Altlarda kalıyorlarsa ciddi bir sorun vardır.

> Bu moleküllerin bir kısmı eğitim setinde de bulunabilir; o durumda yüksek
> tahmin almaları zaten beklenir. Asıl ilginç olan, eğitimde **olmayan**
> bilinen inhibitörlerin de yüksek sıralanmasıdır.

In [ ]:
BILINEN_NTRK = ["larotrectinib", "entrectinib", "repotrectinib",
                "selitrectinib", "crizotinib", "lestaurtinib", "altiratinib",
                "sitravatinib", "cabozantinib", "milciclib"]

sonuc["siralama"] = sonuc[f"tahmin_{HEDEF_KOLON}"].rank(ascending=False).astype(int)

isim_kucuk = sonuc["Isim"].fillna("").str.lower()
bulunan = sonuc[isim_kucuk.isin(BILINEN_NTRK)].copy()

if len(bulunan):
    print(f"Kutuphanede bulunan bilinen NTRK/coklu-kinaz inhibitorleri "
          f"({len(bulunan)} adet)")
    print("=" * 104)
    gosterilecek = ["Isim", "siralama", f"tahmin_{HEDEF_KOLON}", "tahmin_nM",
                    "aktif_olasiligi", "AD_icinde", "en_yuksek_tanimoto",
                    "Egitimde_var"]
    print(bulunan.sort_values("siralama")[gosterilecek].round(3)
          .to_string(index=False))

    print(f"\nToplam kutuphane molekulu : {len(sonuc)}")
    print(f"Bu ilaclarin ortalama siralamasi : "
          f"{bulunan['siralama'].mean():.0f}. "
          f"(ust %{100 * bulunan['siralama'].mean() / len(sonuc):.1f})")
    ilk_yuzde10 = int((bulunan["siralama"] <= len(sonuc) * 0.1).sum())
    print(f"Ilk %10'da yer alan             : {ilk_yuzde10} / {len(bulunan)}")
    if ilk_yuzde10 >= max(1, len(bulunan) // 2):
        print("\n-> BASARILI: bilinen inhibitorlerin cogu ust dilimde.")
        print("   Model gercek bir yapi-aktivite iliskisi ogrenmis.")
    else:
        print("\n-> DIKKAT: bilinen inhibitorler bekledigimiz kadar yukarida degil.")
        print("   Modelin veya AD sinirlarinin gozden gecirilmesi gerekir.")
else:
    print("Kutuphanede isimle eslesen bilinen NTRK inhibitoru bulunamadi.")
    print("(Kutuphane kismi indirilmis olabilir ya da isimler farkli yazilmis.)")

In [ ]:
# Bilinen inhibitorler kutuphanenin genelinde nerede duruyor?
fig, eksen = plt.subplots(figsize=(11, 4.6))

eksen.hist(sonuc[f"tahmin_{HEDEF_KOLON}"], bins=45, color="#95a5a6",
           edgecolor="white", alpha=0.85,
           label=f"Butun kutuphane (n={len(sonuc)})")

if len(bulunan):
    for _, satir in bulunan.iterrows():
        eksen.axvline(satir[f"tahmin_{HEDEF_KOLON}"], color="#c0392b",
                      linewidth=1.6, alpha=0.8)
    eksen.axvline(bulunan[f"tahmin_{HEDEF_KOLON}"].iloc[0], color="#c0392b",
                  linewidth=1.6, label="bilinen NTRK inhibitorleri")

if ESIK is not None:
    eksen.axvline(ESIK, color="#27ae60", linestyle="--", linewidth=2,
                  label=f"aktiflik esigi ({HEDEF_KOLON} = {ESIK})")
eksen.set_xlabel(f"Tahmin edilen {HEDEF_KOLON}")
eksen.set_ylabel("Molekul sayisi")
eksen.set_title("Tarama kutuphanesinin tahmin dagilimi ve bilinen inhibitorler")
eksen.legend(fontsize=9)
eksen.grid(alpha=0.3, axis="y", linestyle="--")
plt.tight_layout()
plt.savefig(GORSEL / "02_tarama_dagilimi.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 7. Adayları gruplamak ve sıralamak

Molekülleri tek bir süzgeçten geçirip gerisini atmak yerine **dört gruba**
ayırıyoruz. Böylece hiçbir bilgi kaybolmaz ve her molekülün neden listede olup
olmadığı görünür:

| Grup | Ölçüt | Anlamı |
|---|---|---|
| **A** | AD içinde **ve** sınıflandırıcı "aktif" dedi | Birincil adaylar — iki model de aynı şeyi söylüyor |
| **B** | AD içinde, sınıflandırıcı "aktif" demedi | İkincil adaylar — modeller uyuşmuyor, dikkatle bakın |
| **C** | AD dışında | Tahmine güvenilmez |
| **D** | Eğitim setinde zaten var | Yeni bir bulgu değil (ama sağlık kontrolü için değerli) |

Aday listesini **A grubundan** kuruyoruz, tahmin edilen potense göre sıralı.
A grubu `TOP_N` sayısını doldurmazsa listeyi **B grubundan** tamamlıyoruz ve
bunu açıkça belirtiyoruz — her adayın hangi gruptan geldiği tabloda ve SDF
alanlarında yazılı.

> **Neden iki model uyuşmayabilir?** Regresyon yüksek potens tahmin ederken
> sınıflandırıcı düşük olasılık veriyorsa, molekül karar sınırına yakındır.
> Bu bir çelişki değil, bir **belirsizlik işaretidir**.

In [ ]:
siralama_kolonu = f"tahmin_{HEDEF_KOLON}"

# ── Her molekulu bir gruba yerlestir ─────────────────────────────────────────
sonuc["Grup"] = "C"                                      # AD disinda
ad_icinde = sonuc["AD_icinde"].fillna(False).astype(bool)
sonuc.loc[ad_icinde, "Grup"] = "B"                       # AD icinde
if paket_snf is not None:
    aktif_dedi = (sonuc["siniflandirici_aktif"] == 1)
    sonuc.loc[ad_icinde & aktif_dedi, "Grup"] = "A"      # + siniflandirici onayi
else:
    sonuc.loc[ad_icinde, "Grup"] = "A"
sonuc.loc[sonuc["Egitimde_var"] == 1, "Grup"] = "D"      # zaten biliniyor

GRUP_ACIKLAMA = {
    "A": "AD icinde + siniflandirici 'aktif' dedi  (birincil adaylar)",
    "B": "AD icinde, siniflandirici 'aktif' demedi (ikincil adaylar)",
    "C": "AD disinda — tahmine guvenilmez",
    "D": "Egitim setinde zaten var — yeni bulgu degil",
}

ozet_grup = (sonuc.groupby("Grup")
             .agg(Molekul=(siralama_kolonu, "size"),
                  En_yuksek_tahmin=(siralama_kolonu, "max"),
                  Ortalama_tahmin=(siralama_kolonu, "mean"))
             .reindex(["A", "B", "C", "D"]).fillna(0))
ozet_grup["Aciklama"] = [GRUP_ACIKLAMA[g] for g in ozet_grup.index]

print("ADAY GRUPLARI")
print("=" * 96)
print(ozet_grup.round(3).to_string())

# ── Siralama: once A grubu, gerekirse B ile tamamla ──────────────────────────
a_grubu = sonuc[sonuc["Grup"] == "A"].sort_values(siralama_kolonu, ascending=False)
b_grubu = sonuc[sonuc["Grup"] == "B"].sort_values(siralama_kolonu, ascending=False)

top = a_grubu.head(TOP_N)
if len(top) < TOP_N and B_ILE_TAMAMLA and len(b_grubu):
    eksik = TOP_N - len(top)
    top = pd.concat([top, b_grubu.head(eksik)])
    print(f"\nNot: A grubunda {len(a_grubu)} molekul var; liste B grubundan "
          f"{min(eksik, len(b_grubu))} molekulle tamamlandi.")
top = top.reset_index(drop=True)

print()
if len(top) == 0:
    print("!! Hicbir molekul AD icinde kalmadi.")
    print("   Oturum 5'te TANIMOTO_ESIGI degerini dusurup yeniden calistirin.")
else:
    print(f"SECILEN {len(top)} ADAY")
    print("=" * 120)
    gosterilecek = ["Grup", "molecule_chembl_id", "Isim", "max_phase",
                    siralama_kolonu, "tahmin_nM", "aktif_olasiligi",
                    "en_yuksek_tanimoto", "kaldirac"]
    gosterilecek = [k for k in gosterilecek if k in top.columns]
    print(top[gosterilecek].round(3).to_string(index=True))

    if "max_phase" in top.columns:
        print()
        print("Secilen adaylarin klinik faz dagilimi")
        print("-" * 44)
        for faz, n in top["max_phase"].value_counts().sort_index().items():
            print(f"  Faz {int(faz)}: {n:>3} molekul")

In [ ]:
# Adaylarin yapilari
if len(top):
    n_goster = min(12, len(top))
    moller_top, etiketler = [], []
    for i in range(n_goster):
        mol = Chem.MolFromSmiles(top.loc[i, "Smiles"])
        if mol is None:
            continue
        isim = top.loc[i, "Isim"] or top.loc[i, "molecule_chembl_id"]
        isim = str(isim)[:26]
        moller_top.append(mol)
        etiketler.append(f"{i + 1}. {isim}\n"
                         f"{HEDEF_KOLON} {top.loc[i, f'tahmin_{HEDEF_KOLON}']:.2f} "
                         f"({top.loc[i, 'tahmin_nM']:.0f} nM)")

    display(Draw.MolsToGridImage(moller_top, legends=etiketler, molsPerRow=3,
                                 subImgSize=(300, 250)))

---
## 8. SDF dosyalarını üretmek

Şimdi asıl çıktıyı üretiyoruz. Her molekül için SDF'ye şu bilgileri
yazacağız:

| SDF alanı | İçerik |
|---|---|
| `Isim`, `ChEMBL_ID` | Kimlik |
| `Tahmin_pIC50`, `Tahmin_nM` | Regresyon tahmini |
| `Aktif_olasiligi` | Sınıflandırıcı çıktısı |
| `AD_icinde`, `Kaldirac`, `En_yuksek_Tanimoto` | Güven bilgisi |
| `En_benzer_egitim_molekulu` | Hangi bilinen moleküle benziyor |
| `Model`, `Hedef`, `Uretim_tarihi` | İzlenebilirlik |

**3B koordinat üretimi.** SMILES yalnızca bağlantı bilgisi taşır; doklama için
3B konformasyon gerekir. İki adımda üretiyoruz:

1. **ETKDG** (RDKit'in mesafe geometrisi algoritması) ile başlangıç
   konformasyonu üretilir.
2. **MMFF94** kuvvet alanı ile geometri optimize edilir.

> ⚠️ **Önemli sınır.** Bu, tek ve *keyfi* bir düşük enerjili konformasyondur —
> proteine bağlanan gerçek konformasyon olduğu garanti **değildir**. Ciddi bir
> doklama çalışmasında molekül başına çok sayıda konformer üretilir ya da
> doklama programının kendi arama algoritması kullanılır.

In [ ]:
from datetime import date


def uc_boyutlu_yap(mol, tohum=RASTGELE_TOHUM):
    """ETKDG ile 3B konformasyon uretir ve MMFF94 ile optimize eder."""
    mol_h = Chem.AddHs(mol)
    parametreler = AllChem.ETKDGv3()
    parametreler.randomSeed = tohum
    if AllChem.EmbedMolecule(mol_h, parametreler) != 0:
        # Basarisiz olursa rastgele koordinatlarla tekrar dene
        parametreler.useRandomCoords = True
        if AllChem.EmbedMolecule(mol_h, parametreler) != 0:
            return None
    try:
        AllChem.MMFFOptimizeMolecule(mol_h, maxIters=500)
    except Exception:
        pass            # optimizasyon basarisiz olsa da gomme kullanilabilir
    return mol_h


def ozellikleri_yaz(mol, satir, sira):
    """Tahmin ve guven bilgilerini SDF alanlari olarak molekule ekler."""
    mol.SetProp("_Name", str(satir.get("Isim")
                             or satir.get("molecule_chembl_id")
                             or f"aday_{sira}"))
    alanlar = {
        "Sira": sira,
        "Aday_grubu": satir.get("Grup", "?"),
        "Klinik_faz": (int(satir["max_phase"])
                       if pd.notna(satir.get("max_phase")) else "bilinmiyor"),
        "ChEMBL_ID": satir.get("molecule_chembl_id"),
        "Isim": satir.get("Isim"),
        "Smiles": satir.get("Smiles"),
        f"Tahmin_{HEDEF_KOLON}": round(float(satir[f"tahmin_{HEDEF_KOLON}"]), 3),
        "Tahmin_nM": round(float(satir["tahmin_nM"]), 2),
        "Aktif_olasiligi": (round(float(satir["aktif_olasiligi"]), 4)
                            if pd.notna(satir.get("aktif_olasiligi")) else "yok"),
        "AD_icinde": "evet" if satir.get("AD_icinde") else "hayir",
        "Kaldirac": round(float(satir["kaldirac"]), 5),
        "Kaldirac_kritik": round(float(h_kritik), 5),
        "En_yuksek_Tanimoto": round(float(satir["en_yuksek_tanimoto"]), 4),
        "En_benzer_egitim_molekulu": satir.get("en_benzer_egitim_smiles"),
        "Egitim_setinde_var": "evet" if satir.get("Egitimde_var") else "hayir",
        "Hedef": f"{TARGET_ADI} (CHEMBL2815)",
        "Model": f"{paket_reg['model_adi']} / {paket_reg['ozellik_seti']}",
        "Model_test_R2": round(float(paket_reg["test_r2"]), 4),
        "Uretim_tarihi": date.today().isoformat(),
        "Not": ("Kurs alistirmasi kapsaminda uretilmis MODEL TAHMINIDIR; "
                "deneysel dogrulama yapilmamistir."),
    }
    for anahtar, deger in alanlar.items():
        if deger is not None:
            mol.SetProp(anahtar, str(deger))
    return mol


print("Hazir.")

In [ ]:
if len(top) == 0:
    print("Aday yok; SDF uretimi atlandi.")
else:
    print(f"{len(top)} aday icin SDF uretiliyor"
          f"{' (3B koordinatli)' if UC_BOYUT else ' (2B)'}...")

    hazir_moller, basarisiz = [], []
    baslangic = time.time()

    for i, satir in top.iterrows():
        mol = Chem.MolFromSmiles(satir["Smiles"])
        if mol is None:
            basarisiz.append((i, "SMILES ayristirilamadi"))
            continue

        if UC_BOYUT:
            mol_3b = uc_boyutlu_yap(mol)
            if mol_3b is None:
                basarisiz.append((i, "3B gomme basarisiz"))
                # 2B ile devam et
                AllChem.Compute2DCoords(mol)
            else:
                mol = mol_3b
        else:
            AllChem.Compute2DCoords(mol)

        hazir_moller.append(ozellikleri_yaz(mol, satir, i + 1))

    print(f"  {len(hazir_moller)} molekul hazir, {time.time() - baslangic:.1f} s")
    if basarisiz:
        print(f"  Sorunlu: {len(basarisiz)}")
        for i, neden in basarisiz[:5]:
            print(f"    sira {i + 1}: {neden}")

### Tek birleşik SDF

In [ ]:
if len(top) and hazir_moller:
    yol_birlesik = SDF_KLASOR / f"ntrk1_adaylar_top{len(hazir_moller)}.sdf"

    yazici = Chem.SDWriter(str(yol_birlesik))
    for mol in hazir_moller:
        yazici.write(mol)
    yazici.close()

    boyut_kb = yol_birlesik.stat().st_size / 1024
    print(f"Birlesik SDF yazildi")
    print(f"  Dosya   : {yol_birlesik.name}")
    print(f"  Molekul : {len(hazir_moller)}")
    print(f"  Boyut   : {boyut_kb:.1f} KB")
    print(f"  Konum   : {SDF_KLASOR}")

    # Dogrulama: geri okuyup alanlarin yazildigini kontrol et
    geri = [m for m in Chem.SDMolSupplier(str(yol_birlesik)) if m is not None]
    print(f"\nDogrulama: {len(geri)} molekul geri okundu.")
    if geri:
        ornek = geri[0]
        print(f"  Ilk molekulun adi : {ornek.GetProp('_Name')}")
        print(f"  Konformasyon      : "
              f"{'3B' if ornek.GetNumConformers() and ornek.GetConformer().Is3D() else '2B'}")
        print(f"  Yazilan alan sayisi: {len(ornek.GetPropNames())}")
        print("\n  Ornek alanlar:")
        for anahtar in list(ornek.GetPropNames())[:10]:
            deger = str(ornek.GetProp(anahtar))
            print(f"    {anahtar:<28} {deger[:52]}")

### Molekül başına ayrı SDF dosyaları

Her molekülü kendi dosyasına yazıyoruz. Dosya adını `sıra_isim` biçiminde
kuruyor ve dosya sistemi için güvenli hâle getiriyoruz (boşluk, eğik çizgi gibi
karakterler sorun çıkarır).

Bu biçim özellikle **paralel doklama** işlerinde kullanışlıdır: her dosya ayrı
bir işe verilebilir ve sonuçlar dosya adıyla izlenebilir.

In [ ]:
def guvenli_ad(metin, uzunluk=42):
    """Dosya adi icin guvenli bir dizgi uretir."""
    izinli = []
    for karakter in str(metin):
        if karakter.isalnum() or karakter in "-_":
            izinli.append(karakter)
        elif karakter in " .":
            izinli.append("_")
    temiz = "".join(izinli).strip("_")
    return (temiz[:uzunluk] or "molekul")


if len(top) and hazir_moller:
    # Onceki calistirmalardan kalan dosyalari temizle
    for eski in SDF_TEK_TEK.glob("*.sdf"):
        eski.unlink()

    yazilanlar = []
    for sira, mol in enumerate(hazir_moller, start=1):
        satir = top.iloc[sira - 1]
        etiket = guvenli_ad(satir.get("Isim") or satir.get("molecule_chembl_id"))
        dosya_adi = f"{sira:02d}_{etiket}.sdf"
        yol = SDF_TEK_TEK / dosya_adi

        yazici = Chem.SDWriter(str(yol))
        yazici.write(mol)
        yazici.close()
        yazilanlar.append((dosya_adi, yol.stat().st_size / 1024))

    print(f"{len(yazilanlar)} ayri SDF dosyasi yazildi")
    print(f"Konum: {SDF_TEK_TEK}")
    print("-" * 62)
    for ad, kb in yazilanlar[:12]:
        print(f"  {ad:<48} {kb:>6.1f} KB")
    if len(yazilanlar) > 12:
        print(f"  ... ve {len(yazilanlar) - 12} dosya daha")

### Hepsini tek arşivde toplamak

Çok sayıda küçük dosyayı Drive'dan tek tek indirmek zahmetlidir. Bir **ZIP**
arşivi oluşturuyoruz.

In [ ]:
if len(top) and hazir_moller:
    yol_zip = SDF_KLASOR / f"ntrk1_adaylar_sdf.zip"

    with zipfile.ZipFile(yol_zip, "w", zipfile.ZIP_DEFLATED) as arsiv:
        arsiv.write(yol_birlesik, arcname=yol_birlesik.name)
        for dosya in sorted(SDF_TEK_TEK.glob("*.sdf")):
            arsiv.write(dosya, arcname=f"molekul_basina/{dosya.name}")

    print(f"Arsiv olusturuldu: {yol_zip.name}")
    print(f"  Boyut   : {yol_zip.stat().st_size / 1024:.1f} KB")
    with zipfile.ZipFile(yol_zip) as arsiv:
        print(f"  Icerik  : {len(arsiv.namelist())} dosya")

    # Colab'da dogrudan indirmek isterseniz:
    # from google.colab import files
    # files.download(str(yol_zip))
    print("\nColab'da bilgisayariniza indirmek icin alttaki satirlari "
          "calistirin:")
    print("  from google.colab import files")
    print(f"  files.download('{yol_zip}')")

---
## 9. Bütün tahminleri CSV olarak kaydetmek

SDF yalnızca seçilmiş adayları içeriyor. Bütün kütüphanenin tahminlerini de
kaydediyoruz: ileride farklı eşiklerle yeniden süzmek isteyebilirsiniz.

In [ ]:
kolonlar = ["Grup", "molecule_chembl_id", "Isim", "max_phase",
            "Smiles", "siralama",
            f"tahmin_{HEDEF_KOLON}", "tahmin_nM", "aktif_olasiligi",
            "siniflandirici_aktif", "AD_icinde", "kaldirac", "kaldirac_ok",
            "en_yuksek_tanimoto", "tanimoto_ok", "en_benzer_egitim_smiles",
            "Egitimde_var"]
kolonlar = [k for k in kolonlar if k in sonuc.columns]

yol_tum = BU / "ntrk1_kutuphane_tahminleri.csv"
sonuc[kolonlar].sort_values("siralama").to_csv(yol_tum, index=False)

yol_adaylar = BU / "ntrk1_secilen_adaylar.csv"
if len(top):
    top[kolonlar].to_csv(yol_adaylar, index=False)

print("Kaydedilen dosyalar")
print("-" * 76)
for yol in [yol_tum, yol_adaylar]:
    if yol.exists():
        print(f"  {yol.name:<46} {yol.stat().st_size / 1024:>8.1f} KB")
if len(top) and hazir_moller:
    for yol in [yol_birlesik, yol_zip]:
        print(f"  {yol.name:<46} {yol.stat().st_size / 1024:>8.1f} KB")
    print(f"  molekul_basina/ ({len(list(SDF_TEK_TEK.glob('*.sdf')))} SDF dosyasi)")

print(f"\nBu oturum  : {BU}")
print(f"SDF klasoru: {SDF_KLASOR}")

---
## 10. Sonuçları nasıl yorumlamalı? — ve nasıl yorumlamamalı

Bu, kursun en önemli bölümü olabilir.

### Elimizde ne var?

Bir model, klinik aşama ilaç kütüphanesini tarayıp bazı moleküller için yüksek
`pIC50` tahmin etti ve bunlar uygulanabilirlik alanı içinde kaldı.
Bilinen NTRK inhibitörlerinin üst sıralarda çıkması, modelin anlamlı bir şey
öğrendiğini destekliyor.

### Bu ne **değildir**?

> **Bunlar ilaç adayı değildir. Deneysel olarak doğrulanmamış model
> tahminleridir.**

Aradaki farkı bilmek, hesaplamalı bir çalışmayı sorumlu biçimde yürütmenin
temelidir. Üretilen listenin sınırları:

1. **Deneysel doğrulama yok.** Bir tahmin, bir ölçüm değildir. Bu listenin tek
   meşru kullanımı, deneysel olarak test edilecek moleküllerin sırasını
   belirlemektir — sonucu ilan etmek değil.
2. **Model yalnızca bağlanma potensini tahmin eder.** Bir ilacın işe yaraması
   için ayrıca emilmesi, dağılması, metabolize olmaması, toksik olmaması ve
   hedefe *in vivo* ulaşması gerekir. Hiçbirini modellemedik.
3. **Seçicilik modellenmedi.** Bir molekül NTRK1'e bağlanırken onlarca başka
   kinaza da bağlanabilir. Seçicilik, kinaz inhibitörü geliştirmede en zor
   sorundur ve ayrı modeller gerektirir.
4. **AD içinde olmak doğruluk garantisi değildir.** Yalnızca "bu molekül
   eğitim verisine benziyor, dolayısıyla tahmin makul ölçüde güvenilir"
   demektir.
5. **Klinik aşamadaki moleküller zaten yoğun biçimde çalışılmıştır.** Gerçekten güçlü
   ve bilinmeyen bir NTRK inhibitörünün bu kütüphanede saklı kalması pek
   olası değildir. Alıştırmanın değeri yöntemdedir, bulgularda değil.

### Gerçek bir çalışmada bundan sonra ne yapılır?

1. Adayları **doklama** ile değerlendirmek (ürettiğimiz SDF dosyaları buna hazır)
2. Bağlanma modunu ve kararlılığını **moleküler dinamik** ile incelemek
3. En umut veren birkaç molekül için **deneysel biyokimyasal test** (enzim
   inhibisyon tayini)
4. Deneysel sonuçları veri setine ekleyip modeli **yeniden eğitmek** — buna
   **aktif öğrenme** (active learning) denir ve modern ilaç keşfinin
   işleyiş biçimidir.

Dördüncü madde, bu kursun döngüsünü kapatır: model deneyi yönlendirir, deney
modeli iyileştirir.

---
## 11. Kursun özeti

### Altı oturumda ne yaptık?

| Oturum | Konu | Ana çıktı |
|---|---|---|
| **1** | Veri hazırlama | Temiz `Smiles → pIC50` tablosu |
| **2** | Özellik üretimi | Betimleyiciler, ECFP4, MACCS |
| **3** | Regresyon | LazyPredict → Optuna → CV → y-randomizasyon |
| **4** | Sınıflandırma | Dengesizlik, ROC/PR, MCC, karar eşiği |
| **5** | Karşılaştırma, XAI, AD | SHAP, Williams grafiği, Tanimoto |
| **6** | Tarama ve çıktı | Sıralı adaylar, SDF dosyaları |

### Kurstan götürülecek on ilke

1. **Veri hazırlığı modelleme kadar önemlidir.** Kanonikleştirmeyi atlamak
   gizli tekrarlar ve veri sızıntısı üretir.
2. **Sansürlü değerleri (`>`, `<`) regresyona sokmayın.**
3. **Ölçekleme modelin parçasıdır** — `Pipeline` içinde olmalı, veriye
   global uygulanmamalı.
4. **Test seti kutsaldır.** Model seçimi ve ayarlaması bitmeden açılmaz.
5. **Rastgele bölünme iyimserdir.** İskelet bazlı bölünme gerçek genelleme
   kapasitesini ölçer.
6. **y-randomizasyon zorunludur.** Modelin öğrendiğinin şans olmadığını
   kanıtlar.
7. **Doğruluğa tek başına güvenmeyin.** Temel çizgiyi bildirin; dengeli
   doğruluk ve MCC kullanın.
8. **Her tahminin bir güven derecesi olmalı** — uygulanabilirlik alanı.
9. **SHAP modelin ne gördüğünü söyler, nedenselliği söylemez.**
10. **Tahmin, ölçüm değildir.** Hesaplamalı sonuçlar deneyi yönlendirir,
    onun yerini almaz.

### Devam etmek isterseniz

- `ILERI_OKUMA.md` dosyasında kaynaklar, makaleler ve açık veri setleri var.
- Kendi hedefinizle çalışmak için: Oturum 1'deki `TARGET_CHEMBL_ID`'yi
  değiştirip bütün not defterlerini sırayla çalıştırmanız yeterli. Kod hedefe
  özel hiçbir varsayım yapmaz.
- Bu depodaki bütün materyal CC BY 4.0 ile paylaşılmıştır; derslerinizde
  kullanabilir, uyarlayabilirsiniz.

---

### Alıştırmalar

1. `KUTUPHANE_KAYNAGI = "kendi"` yapıp kendi ilgilendiğiniz molekülleri
   tarayın. AD içinde kalıyorlar mı?
2. `SADECE_AD_ICINDE = False` yapın. Listeye hangi moleküller giriyor?
   Yapılarına bakın — eğitim setine neden benzemiyorlar?
3. `UC_BOYUT = True` ile üretilen SDF dosyasını PyMOL ya da ChimeraX'te açıp
   konformasyonları inceleyin.
4. İlk 25 aday arasında kimyasal olarak kaç farklı iskelet var? Çeşitlilik
   düşükse, bu sanal tarama için bir sorun mudur? (İpucu: aynı serinin 25
   türevini sentezlemek mi, 25 farklı iskeleti test etmek mi daha bilgilendirici?)
5. Regresyon ve sınıflandırma modellerinin **uyuşmadığı** molekülleri bulun
   (regresyon yüksek, sınıflandırıcı düşük olasılık veya tersi). Bu
   uyuşmazlığın nedeni ne olabilir?